In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys
import itertools as it
import pandas as pd
from datetime import datetime as dt
from datetime import timedelta
from pathlib import Path
from hypo_sleep import Session
import spikeinterface.full as si
import spikeinterface.preprocessing as spp
from scipy import signal, stats
import xarray as xr
import statsmodels.api as sm
import statsmodels.formula.api as smf
from functools import partial
from ghostipy.dsp import signal_envelope
import re
from nitime.utils import dpss_windows

import pdb

In [2]:
data_path = Path("/gpfs01/born/animal/DanielG/hypo_sleep/processed_data/")
full_dict = {
    "none": {
        "dates": [
            "2025-02-18_09-19-26",
            "2025-02-20_08-58-57",
            "2025-03-20_09-01-41",
            "2025-02-24_09-00-29",
            "2025-02-26_09-05-08",
        ],
        "configs": [
            "5d77",
            "790d",
            "8a55",
            "3e4b",
            "d205",
        ],
        "animal": ["HYDO03", "HYDO03", "HYDO03", "HYDO04", "HYDO04"],
    },
    "sleep": {
        "dates": [
            "2025-02-19_08-59-23",
            "2025-03-03_08-55-10",
            "2025-02-25_09-06-23",
            "2025-03-07_08-52-44",
        ],
        "configs": [
            "d628",
            "c687",
            "073a",
            "9080",
        ],
        "animal": ["HYDO03", "HYDO03", "HYDO04", "HYDO04"],
    },
    "food": {
        "dates": [
            "2025-02-21_09-04-29",
            "2025-02-28_09-02-06",
            "2025-03-04_08-55-25",
        ],
        "configs": [
            "a964",
            "0674",
            "bc96",
        ],
        "animal": ["HYDO03", "HYDO03", "HYDO04"],
    },
}
depth_ordered_chs = np.array(
    [
        "24",
        "23",
        "25",
        "22",
        "8",
        "7",
        "9",
        "6",
        "15",
        "0",
        "14",
        "1",
        "26",
        "21",
        "27",
        "20",
        "10",
        "5",
        "28",
        "19",
        "11",
        "4",
        "29",
        "18",
        "12",
        "3",
        "30",
        "17",
        "13",
        "2",
        "31",
        "16",
    ]
)

In [3]:
def mt_specpb(data, Fs=1000, NW=4, chunk_size=None, chunk_avg=False):
    tapers, _ = dpss_windows(data.shape[-1], NW, 2 * NW - 1)  # Compute the tapers,
    tapers *= np.sqrt(Fs)  # ... and scale them.
    if chunk_size is None:
        chunk_size = data.shape[0]

    nchunks = int(np.ceil(data.shape[0] / chunk_size))
    spectra = []
    spectra_sem = []

    for i in range(nchunks):
        chunk = data[i * chunk_size : (i + 1) * chunk_size, :]
        # Taper and FFT
        dataT = np.array(
            [[trial * t for t in tapers] for trial in chunk]
        )  # shape: (nchunk, k, time)
        T = np.fft.rfft(tapers, axis=-1)  # shape: (k, nf)
        J = np.fft.rfft(dataT, axis=-1)  # shape: (nchunk, k, nf)

        # Subtract DC
        dc = np.array([T * trial.mean() for trial in chunk])  # shape: (nchunk, k, nf)
        J -= dc

        # Spectrum: power
        J *= J.conj()  # power
        S_chunk = J.mean(1).real
        spectra.append(np.mean(S_chunk, axis=0))  # mean across chunk
        spectra_sem.append(stats.sem(S_chunk, axis=0))
    spectra = np.stack(spectra)  # shape: (nchunks, nf)
    spectra_sem = np.stack(spectra_sem)
    f = np.fft.rfftfreq(data.shape[-1], 1 / Fs)
    if chunk_avg:
        spectra = spectra.mean(0)  # Average across trials.
        spectra_sem = spectra_sem.mean(0)
    return f, spectra, spectra_sem

In [ ]:
states = ["wake-all", "nrem-all", "rem-all"]
data_path = Path("/gpfs01/born/animal/DanielG/hypo_sleep/processed_data/")
bin_size = "5ms"
[time_int] = re.findall(r"\d+", bin_size)

cond = "none"
meta_dict = full_dict[cond]
mua_epoch_dict = {
    state: {animal: [] for animal in ["HYDO03", "HYDO04"]} for state in states
}
for animal, date, config_id in zip(
    meta_dict["animal"], meta_dict["dates"], meta_dict["configs"]
):
    date_dt = dt.strptime(date, "%Y-%m-%d_%H-%M-%S")
    base_dt_64 = pd.to_datetime(date_dt, unit="ns")
    for trigger in states:
        mua_id = (
            "mua_4-5thresh_4s_baa2"  # if "peak" in trigger else "mua_4-5thresh_2s_6892"
        )
        xr_path = Path(
            data_path,
            animal,
            date,
            config_id,
            "mua",
            f"5ms_resamp_mua_{trigger}_39_{mua_id}_{config_id}",
        )
        if xr_path.exists():
            files = list(xr_path.glob("*.nc"))
            if len(files) < 1:
                print(f"{xr_path} is empty")
            else:
                tmp_xrs = []
                for file in files:
                    tmp_xrs.append(xr.load_dataarray(file, engine="h5netcdf"))
                    # time_slice = slice(np.timedelta64(-2, "s"), np.timedelta64(2, "s"))
                mua_epoch_dict[trigger][animal].extend(xr.concat(tmp_xrs, dim="time"))

/gpfs01/born/animal/DanielG/hypo_sleep/processed_data/HYDO03/2025-03-20_09-01-41/8a55/mua/5ms_resamp_mua_rem-all_39_mua_4-5thresh_4s_baa2_8a55 is empty


In [ ]:
config_path = Path("/gpfs01/born/animal/DanielG/hypo_sleep/processed_data/")
session_dict = {}
animal_dict = full_dict["none"]
for animal, date, config_id in zip(
    animal_dict["animal"], animal_dict["dates"], animal_dict["configs"]
):
    session_dict[date] = Session(
        config_path, animal, date, config_id, **{"base_path": "remote"}
    )

In [ ]:
epoch_state_dict = {
    state: {animal: [] for animal in set(animal_dict["animal"])} for state in states
}
for date, states in state_data_dict["none"].items():
    date_dt = dt.strptime(date, "%Y-%m-%d_%H-%M-%S")
    base_dt_64 = pd.to_datetime(date_dt, unit="ns")
    animal = [
        animal
        for animal, tmp_date in zip(
            full_dict["none"]["animal"], full_dict["none"]["dates"]
        )
        if tmp_date == date
    ][0]
    for state, data in states.items():
        if data is not None:
            data = data.sortby("time")
            state_key = state.split("-")[0].upper()
            times = session_dict[date].state_dict[state_key]["times"].T
            starts = pd.to_timedelta(times[:, 0], unit="s")
            stops = pd.to_timedelta(times[:, 1], unit="s")
            xr_epochs = []
            for epoch, (start, stop) in enumerate(zip(starts, stops)):

                # delta_times = pd.to_timedelta([start, stop], unit="s")
                tmp_data = data.sel(time=slice(base_dt_64 + start, base_dt_64 + stop))
                if tmp_data.time.size > 0:
                    xr_epochs.append(tmp_data)
                else:
                    print(f"{date} {state} {epoch}: empty slice")

            epoch_state_dict[state][animal].extend(xr_epochs)

In [ ]:
xr.concat(mua_epoch_dict["nrem-all"]["HYDO03"][0], dim="epoch")

<xarray.DataArray 'raw_mua_counts' (epoch: 3, channel: 32, time: 3740902)> Size: 3GB
array([[[ 0.,  0.,  0., ..., nan, nan, nan],
        [ 0.,  0.,  0., ..., nan, nan, nan],
        [ 0.,  0.,  0., ..., nan, nan, nan],
        ...,
        [ 0.,  1.,  0., ..., nan, nan, nan],
        [ 0.,  0.,  0., ..., nan, nan, nan],
        [ 0.,  0.,  0., ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.]]], shape=(3, 32, 3740902))
Coordinates:
  * channel  (channel) <U2 256B '26' '27' '10' '28' '11' ... '19' '5' '20' '21'
  * time     (time) datetime64[ns] 30MB 2025-02-18T10:05:06.995000 ... 2025-0...
Dimensions without coordinates: epoch

In [ ]:
avg_spectra = {}
for event, event_mua in mua_epoch_dict.items():
    f_list = []
    spectra = []
    spectra_sem = []
    avg_spectra[event] = {}
    for tmp_mua in event_mua:
        tmp_trials = tmp_mua.to_numpy().reshape(-1, tmp_mua.shape[-1])
        f, S, S_sem = mt_specpb(
            tmp_trials, Fs=200, NW=4, chunk_size=100, chunk_avg=False
        )
        f_list.append(f)
        spectra.append(S)
        spectra_sem.append(S_sem)

In [ ]:
bin_size = "5ms"
[time_int] = re.findall(r"\d+", bin_size)

mua_epoch_dict = {state: [] for state in states}
for animal, animal_meta in mua_meta_dict.items():
    for date, config_id in zip(animal_meta["dates"], animal_meta["configs"]):
        date_dt = dt.strptime(date, "%Y-%m-%d_%H-%M-%S")
        base_dt_64 = pd.to_datetime(date_dt, unit="ns")
        for trigger in states:
            mua_id = (
                "mua_4-5thresh_4s_baa2"
                if "peak" in trigger
                else "mua_4-5thresh_2s_6892"
            )
            xr_path = Path(
                data_path,
                animal,
                date,
                config_id,
                "mua",
                f"5ms_resamp_mua_{trigger}_39_{mua_id}_{config_id}",
            )
            if xr_path.exists():
                files = list(xr_path.glob("*.nc"))
                if len(files) < 1:
                    print(f"{load_path} is empty")
                else:
                    tmp_xrs = []
                    for file in files:
                        tmp_xrs.append(xr.load_dataarray(file, engine="h5netcdf"))
                        # time_slice = slice(np.timedelta64(-2, "s"), np.timedelta64(2, "s"))
                    mua_epoch_dict[trigger].append(xr.concat(tmp_xrs, dim="time"))
                # resamp_mua = xr.load_dataarray(
                #     xr_path,
                #     engine="h5netcdf",
                # )
                # resamp_mua.name = (
                #     f"resamp_mua_{animal}_{date}_{trigger}_{config_id}"
                # )

                # mua_epoch_dict[trigger].append(
                #     resamp_mua.sel(time=time_slice)  # , channel=ch_slice)
                # )
avg_spectra = {}
for event, event_mua in mua_epoch_dict.items():
    f_list = []
    spectra = []
    spectra_sem = []
    avg_spectra[event] = {}
    for tmp_mua in event_mua:
        tmp_trials = tmp_mua.to_numpy().reshape(-1, tmp_mua.shape[-1])
        f, S, S_sem = mt_specpb(
            tmp_trials, Fs=200, NW=4, chunk_size=100, chunk_avg=False
        )
        f_list.append(f)
        spectra.append(S)
        spectra_sem.append(S_sem)
# TODO: convert to xarray
#     avg_spectra[event]["freqs"] = np.stack(f_list, axis=0)
#     avg_spectra[event]["sem"] = np.concatenate(spectra_sem, axis=0)  # .mean(axis=0)
#     avg_spectra[event]["spectra"] = np.concatenate(spectra, axis=0)  # .mean(axis=0)
# for key, data in avg_spectra.items():
#     np.savez(Path(data_path, "mua", f"avg_psd_{key}_peri.npz"), **data)